In [0]:
%pip install databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()

# Prepare features for ML
def create_provider_ml_features():
    df = spark.table("health.default.gold_provider_fraud_risk")
    
    feature_df = df.select(
        "Provider",
        "total_claims",
        "unique_patients",
        "total_reimbursement",
        "avg_claim_amount",
        "claim_amount_stddev",
        "claims_per_patient",
        "unique_physicians",
        "inpatient_ratio",
        "reimbursement_zscore",
        "cpp_zscore",
        "unique_diagnoses",
        "diagnosis_diversity_ratio",
        "is_fraud"  # Target variable
    ).fillna(0)
    
    return feature_df

df_provider_features = create_provider_ml_features()

# Create feature table
fs.create_table(
    name="health.default.provider_fraud_features",
    primary_keys=["Provider"],
    df=df_provider_features,
    description="Provider behavioral features for fraud detection"
)

In [0]:
def create_beneficiary_ml_features():
    df = spark.table("health.default.silver_beneficiary_profiles")
    
    feature_df = df.select(
        "BeneID",
        "age",
        "Gender",
        "chronic_condition_count",
        "total_claims",
        "unique_providers",
        "total_costs",
        "inpatient_visits",
        "outpatient_visits",
        "risk_score",
        "IPAnnualReimbursementAmt",
        "OPAnnualReimbursementAmt"
    ).fillna(0)
    
    return feature_df

df_bene_features = create_beneficiary_ml_features()

fs.create_table(
    name="health.default.beneficiary_features",
    primary_keys=["BeneID"],
    df=df_bene_features,
    description="Beneficiary demographic and utilization features"
)